# **IMPORT MODEL DARI STAGE 3**

In [87]:
import os

print(os.getcwd())

d:\Rakamin\Final Project\TalentMatchAI


In [88]:
import joblib
import pandas as pd
import numpy as np

In [89]:
import joblib

model = joblib.load("models/gradient_boosting_model.pkl")

print(type(model))

<class 'sklearn.ensemble._gb.GradientBoostingClassifier'>


In [90]:
import joblib

scaler = joblib.load("models/scaler.pkl")

encoders = joblib.load("models/label_encoders.pkl")

feature_order = joblib.load("models/feature_order.pkl")

skill_columns = joblib.load("models/skill_columns.pkl")

print(type(scaler))

print(encoders.keys())

print(feature_order)

print(skill_columns)

<class 'sklearn.preprocessing._data.StandardScaler'>
dict_keys(['education_level', 'internal_referral', 'hiring_eligibility', 'seniority_tier', 'exp_level'])
['education_level', 'years_experience', 'num_relevant_skills', 'internal_referral', 'interview_score', 'technical_test_score', 'soft_skill_score', 'seniority_tier', 'Cloud Computing', 'Communication', 'Data Visualization', 'Excel', 'Leadership', 'Machine Learning', 'Python', 'SQL']
['Cloud Computing', 'Communication', 'Data Visualization', 'Excel', 'Leadership', 'Machine Learning', 'Python', 'SQL']


# **Generate Fitur senior_tier**

In [91]:
import pandas as pd

def create_seniority_tier(years_experience, education_level):
    """
    Membuat exp_level dan seniority_tier
    sesuai pipeline training.
    """

    exp_level = pd.cut(
        [years_experience],
        bins=[-1, 2, 5, 10, 19],
        labels=["Entry", "Junior", "Mid", "Senior"]
    )[0]

    seniority_tier = f"{exp_level}_{education_level}"

    return exp_level, seniority_tier

In [92]:
exp_level, seniority = create_seniority_tier(
    years_experience=7,
    education_level="Bachelor"
)

print(exp_level)
print(seniority)

Mid
Mid_Bachelor


In [93]:
create_seniority_tier(17, "Master")

('Senior', 'Senior_Master')

In [94]:
encoders

tier_encoded = encoders["seniority_tier"].transform([seniority])[0]

print(tier_encoded)

6


In [95]:
print(encoders["seniority_tier"].classes_)

['Entry_Bachelor' 'Entry_Master' 'Entry_PhD' 'Junior_Bachelor'
 'Junior_Master' 'Junior_PhD' 'Mid_Bachelor' 'Mid_Master' 'Mid_PhD'
 'Senior_Bachelor' 'Senior_Master' 'Senior_PhD']


In [96]:
print(
    encoders["seniority_tier"].transform(["Mid_Bachelor"])
)

print(
    encoders["seniority_tier"].transform(["Senior_Master"])
)

[6]
[10]


# **Encode Semua Fitur Kategorikal**

In [97]:
def encode_features(
    education_level,
    internal_referral,
    exp_level,
    seniority_tier
):
    education_encoded = encoders["education_level"].transform(
        [education_level]
    )[0]

    referral_encoded = encoders["internal_referral"].transform(
        [internal_referral]
    )[0]

    exp_encoded = encoders["exp_level"].transform(
        [exp_level]
    )[0]

    tier_encoded = encoders["seniority_tier"].transform(
        [seniority_tier]
    )[0]

    return (
        education_encoded,
        referral_encoded,
        exp_encoded,
        tier_encoded
    )

In [98]:
education_encoded, referral_encoded, exp_encoded, tier_encoded = encode_features(
    education_level="Bachelor",
    internal_referral="No",
    exp_level="Mid",
    seniority_tier="Mid_Bachelor"
)

print("Education :", education_encoded)
print("Referral  :", referral_encoded)
print("Exp Level :", exp_encoded)
print("Tier      :", tier_encoded)

Education : 0
Referral  : 0
Exp Level : 2
Tier      : 6


In [99]:
print(encoders["internal_referral"].classes_)

['No' 'Yes']


In [100]:
print(encoders["education_level"].classes_)

['Bachelor' 'Master' 'PhD']


In [101]:
print(encoders["exp_level"].classes_)

['Entry' 'Junior' 'Mid' 'Senior']


# **Encode Semua Fitur Numerikal**

In [102]:
def create_candidate_dataframe(
    education_encoded,
    years_experience,
    num_relevant_skills,
    referral_encoded,
    interview_score,
    technical_test_score,
    soft_skill_score,
    tier_encoded
):

    df_candidate = pd.DataFrame({
        "education_level":[education_encoded],
        "years_experience":[years_experience],
        "num_relevant_skills":[num_relevant_skills],
        "internal_referral":[referral_encoded],
        "interview_score":[interview_score],
        "technical_test_score":[technical_test_score],
        "soft_skill_score":[soft_skill_score],
        "seniority_tier":[tier_encoded]
    })

    return df_candidate

In [103]:
candidate_df = create_candidate_dataframe(

    education_encoded,

    7,

    2,

    referral_encoded,

    8.5,

    9,

    8,

    tier_encoded

)

candidate_df

,education_level,years_experience,num_relevant_skills,internal_referral,interview_score,technical_test_score,soft_skill_score,seniority_tier
0,0,7,2,0,8.5,9,8,6


# **Skill Dummy**

In [104]:
selected_skills = [
    "Python",
    "SQL"
]

In [105]:
def create_skill_features(selected_skills):

    skill_df = pd.DataFrame(columns=skill_columns)

    skill_df.loc[0] = 0

    for skill in selected_skills:
        if skill in skill_columns:
            skill_df.loc[0, skill] = 1

    return skill_df

In [106]:
skill_df = create_skill_features(
    ["Python", "SQL"]
)

skill_df

,Cloud Computing,Communication,Data Visualization,Excel,Leadership,Machine Learning,Python,SQL
0,0,0,0,0,0,0,1,1


In [107]:
create_skill_features(
    ["Python","Machine Learning","Communication"]
)

,Cloud Computing,Communication,Data Visualization,Excel,Leadership,Machine Learning,Python,SQL
0,0,1,0,0,0,1,1,0


# **Combine Candidate & Skill**

In [108]:
candidate_full = pd.concat(
    [candidate_df, skill_df],
    axis=1
)

candidate_full

,education_level,years_experience,num_relevant_skills,internal_referral,interview_score,technical_test_score,soft_skill_score,seniority_tier,Cloud Computing,Communication,Data Visualization,Excel,Leadership,Machine Learning,Python,SQL
0,0,7,2,0,8.5,9,8,6,0,0,0,0,0,0,1,1


# **REINDEX**

In [109]:
candidate_full = candidate_full.reindex(columns=feature_order)

candidate_full.head()

,education_level,years_experience,num_relevant_skills,internal_referral,interview_score,technical_test_score,soft_skill_score,seniority_tier,Cloud Computing,Communication,Data Visualization,Excel,Leadership,Machine Learning,Python,SQL
0,0,7,2,0,8.5,9,8,6,0,0,0,0,0,0,1,1


In [110]:
print(candidate_full.columns.tolist())

print(feature_order)

['education_level', 'years_experience', 'num_relevant_skills', 'internal_referral', 'interview_score', 'technical_test_score', 'soft_skill_score', 'seniority_tier', 'Cloud Computing', 'Communication', 'Data Visualization', 'Excel', 'Leadership', 'Machine Learning', 'Python', 'SQL']
['education_level', 'years_experience', 'num_relevant_skills', 'internal_referral', 'interview_score', 'technical_test_score', 'soft_skill_score', 'seniority_tier', 'Cloud Computing', 'Communication', 'Data Visualization', 'Excel', 'Leadership', 'Machine Learning', 'Python', 'SQL']


# **SCALING**

In [111]:
candidate_scaled = scaler.transform(candidate_full)

In [112]:
print(candidate_scaled.shape)

(1, 16)


In [117]:
prediction = model.predict(candidate_scaled)

prediction

array([0])

In [120]:
print(model.n_features_in_)

print(feature_order)

16
['education_level', 'years_experience', 'num_relevant_skills', 'internal_referral', 'interview_score', 'technical_test_score', 'soft_skill_score', 'seniority_tier', 'Cloud Computing', 'Communication', 'Data Visualization', 'Excel', 'Leadership', 'Machine Learning', 'Python', 'SQL']


In [121]:
print("Jumlah feature_order:", len(feature_order))

for i, col in enumerate(feature_order):
    print(i, col)

Jumlah feature_order: 16
0 education_level
1 years_experience
2 num_relevant_skills
3 internal_referral
4 interview_score
5 technical_test_score
6 soft_skill_score
7 seniority_tier
8 Cloud Computing
9 Communication
10 Data Visualization
11 Excel
12 Leadership
13 Machine Learning
14 Python
15 SQL


In [122]:
candidate_scaled = scaler.transform(candidate_full)

prediction = model.predict(candidate_scaled)

prediction

array([0])

In [123]:
probability = model.predict_proba(candidate_scaled)

print(probability)

[[0.96656745 0.03343255]]


In [124]:
label = "Eligible" if prediction[0] == 1 else "Not Eligible"

prob = probability[0]

print(f"Hasil Prediksi : {label}")
print(f"Not Eligible  : {prob[0]:.2%}")
print(f"Eligible      : {prob[1]:.2%}")

Hasil Prediksi : Not Eligible
Not Eligible  : 96.66%
Eligible      : 3.34%


# **Refactor Notebook Deployment**

In [125]:
def predict_candidate(
    education_level,
    years_experience,
    num_relevant_skills,
    internal_referral,
    interview_score,
    technical_test_score,
    soft_skill_score,
    selected_skills
):
    ...
    return prediction, probability